In [1]:
import os
os.sys.path.append('/data/scratch/globc/bonassies/workspace/swot_for_flood')

import configparser
from pathlib import Path
from core.swot_project import SwotProject

main_path = "/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio"

# Download and rasterize notebook

This notebook explain how to use the swot_for_flood library to download and rasterize the SWOT data for a given region and time period. 

## Define a project

This library is designed to work with a project. A project is a directory that contains the configuration file `config.cfg` and the data. The configuration file contains the parameters of the project.

In this notebook, we will create a project named "example_download_rasterize" in the directory "examples". The project will contain the SWOT data for the region of interest and time period defined in the configuration file.

In [2]:
config = configparser.ConfigParser()
config.read(main_path + '/config.cfg')

print(type(config),dict(config['CONFIG']))

<class 'configparser.ConfigParser'> {'project': 'Ohio', 'workspace': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples', 'data_path': '/data/scratch/globc/bonassies/data', 'aoi': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/aoi_4326.gpkg', 'floodmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FloodedArea_32616_v2.gpkg', 'controlmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ControlArea_32616.gpkg', 'esa_worldcover_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616.tif', 'crs': '32616', 'aoi_crs': '4326', 'first_time': '2023-01-01', 'last_time': '2025-05-01', 'nodes': '8', 'download_type': 'PIXC', 'passes': '[481]', 'tile_names_selection': '[[481_220L, 481_221L]]', 'list_dry_dates': '[2025-02-20]', 'list_flood_dates': '[2025-02-20]', 'gdal_num_threads': '10', 'gdal_cachemax': '160000', 'do_downl

The config can also be a dictionary

In [3]:
# from pathlib import Path

# param_dict = {
#     'project': 'Greece_EMSR692',
#     'workspace': Path("/data/scratch/globc/bonassies/workspace"),
#     'data_path': Path("/data/scratch/globc/bonassies/data"),
#     'CRS': 'EPSG:32634',
#     'first_time': "2023-09-05",
#     'last_time': "2023-09-20",
#     'aoi': Path("/data/scratch/globc/bonassies/workspace").joinpath('Greece_EMSR692',"aux_data","EMSR692_aois_V2.gpkg"),
#     'aoi_crs': 'EPSG:4326',
#     'passes': [402, 417],
#     'nodes': 8,
#     'do_download': False,
#     'download_type': 'PIXC',
#     'GDAL_NUM_THREADS': 10,
#     'GDAL_CACHEMAX': 160000,
#     'do_make_gpkg': False,
#     'do_make_tiff': False    
# }

Once the config file loaded, we can use it to create a project object. The project object will contain the parameters of the project and the data.

In [3]:
swot_project = SwotProject(config)
print(swot_project)

No automatic download, please use the Downloader object to download the data
Class SWOT_PROJECT():
	project: Ohio
	workspace: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples
	data_path: /data/scratch/globc/bonassies/data
	CRS: 32616
	first_time: 2023-01-01
	last_time: 2025-05-01
	variables: ['sig0', 'coherent_power', 'incidence', 'gamma_tot', 'gamma_SNR', 'gamma_est', 'power_plus_y', 'power_minus_y', 'interf_real', 'interf_imag', 'height', 'classification', 'bright_land_flag']
	tile_names_selection: [['481_220L', '481_221L']]
	list_dry_dates: ['2025-02-20']
	list_flood_dates: ['2025-02-20']
	floodmask_path: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FloodedArea_32616_v2.gpkg
	controlmask_path: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ControlArea_32616.gpkg
	ESA_WC_PATH: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616.tif
	swot_collection: None
	P

On the first time, you should get an error because the data is not downloaded yet. The project object will download the data and store it in the project directory only if do_download is set to True in the configuration file.

You can manually download the data by calling the download method of the project object.

First we need to search for the data we want to download. We can use the search method of the project object to search for the data. The search method will return a list of the data that match the search criteria.

In [5]:
# swot_project.Downloader.search_PIXC()

Then we can download the data by calling the download method of the project object. The download method will download the data and store it in the project directory.

In [6]:
# swot_project.Downloader.download_pool() # if multithreading download
# swot_project.Downloader.download_granules() # if single thread download

Finally we can rasterize the data by calling the rasterize method of the project object. The rasterize method will rasterize the data and store it in the project directory.

First we create gpgk file that combine the netcdf files and then we rasterize the data.

In [4]:
swot_project.Rasterizer.find_pixc()
# swot_project.Rasterizer.pixc_to_gpkg()

In [ ]:
# swot_project.Rasterizer.gpkg_to_tiff()

If needed, we can put extra rasters to the same resolution and extent as the SWOT data. This is useful to compare the SWOT data with other data.

It uses gdal to rasterize the data. gdal is used as command line using os.system.

In [7]:
raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616.tif")
raster_crs = '32616'
interp = 'near'
swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)
raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FABDEM_Fusion_cut_32616.tif")
raster_crs = '32616'
interp = 'bilinear'
swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)
raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FM_20250222T000000_S1_POST_fusion_cut_32616.tif")
raster_crs = '32616'
interp = 'near'
swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)

>>> Converting the AUXILARY rasters to tiff
gdalwarp -s_srs EPSG:32616 -t_srs EPSG:32616 -te 465824.0000002095 4176822.500110473 525844.562499841 4213272.500110996 -ts 6003.0 3646.0 -r near -of GTiff /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616.tif /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616_nrow3646_ncol6003.tif
Creating output file that is 6003P x 3646L.
Processing /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
>>> Converting the AUXILARY rasters to tiff
gdalwarp -s_srs EPSG:32616 -t_srs EPSG:32616 -te 465824.0000002095 4176822.500110473 525844.562499841 4213272.500110996 -ts 6003.0 3646.0 -r bilinear -of GTiff /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FABDEM_Fusion_cut_32616.tif /data/scratch/globc/bonassies/

You can check other notebooks for more information about the library.